In [ ]:
VERSION = "1.1.0_sextractor"

# Sextractor test
# 1.1.0: "4_000",
# 1.1.1: "40_000",
# hypercube + shear round 2 + 10 gal/patch + shear |0.06| + ellips gaussian + pixels_out

# lib

In [ ]:
from pathlib import Path

from jax import numpy
from polars import DataFrame, read_csv
from tqdm import tqdm
from utils import (
    best_results,
    case_results,
    ellips_ab,
    ellips_quad,
    ellips_ratio,
    extra_imgs,
    histos,
    vignettes,
)

DATASET_DIR = Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / VERSION
VERSION_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR") / VERSION
CATALOG_DIR = VERSION_DIR / "CATALOG"
IMAGES_DIR = VERSION_DIR / "IMAGES"
PLOTS_DIR = VERSION_DIR / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# images

In [ ]:
CHOICE = "simu_00002"

extra_imgs(
    [
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        IMAGES_DIR / f"background_{CHOICE}",
        IMAGES_DIR / f"residual_{CHOICE}_1",
        IMAGES_DIR / f"segmentation_{CHOICE}",
    ],
    ["image", "fitting", "background", "residual", "segmentation"],
    PLOTS_DIR,
)

# vignettes

In [ ]:
CHOICE = "simu_00004"

vignettes(
    PLOTS_DIR,
    csv_file=CATALOG_DIR / f"catalog_{CHOICE}.csv",
    vignettes_file=IMAGES_DIR / f"vignettes_{CHOICE}.fits",
)

# inference

In [ ]:
RESULTS_FILES = sorted(list((CATALOG_DIR).glob("catalog_simu_*.csv")))
NB_TOTAL = len(RESULTS_FILES)

detections, avg_e1_ra, avg_e2_ra, avg_e1, avg_e2, avg_e1_ab, avg_e2_ab = [], [], [], [], [], [], []
avg_e1_quad, avg_e2_quad, ell_1, ell_2 = [], [], [], []
for sample in tqdm(RESULTS_FILES, total=NB_TOTAL):
    df = read_csv(sample)
    detections.append(df.shape[0])

    avg_e1.append(df["e1"].to_numpy().mean())
    avg_e2.append(df["e2"].to_numpy().mean())

    e1_ra, e2_ra = ellips_ratio(df["ratio"].to_numpy(), df["angle"].to_numpy())
    avg_e1_ra.append(e1_ra.mean())
    avg_e2_ra.append(e2_ra.mean())

    e1_ab, e2_ab = ellips_ab(
        df["ellipse_a"].to_numpy(), df["ellipse_b"].to_numpy(), df["ellipse_theta"].to_numpy()
    )
    avg_e1_ab.append(e1_ab.mean())
    avg_e2_ab.append(e2_ab.mean())

    e1_quad, e2_quad = ellips_quad(
        df["ellipse_cxx"].to_numpy(), df["ellipse_cyy"].to_numpy(), df["ellipse_cxy"].to_numpy()
    )
    avg_e1_quad.append(e1_ab.mean())
    avg_e2_quad.append(e2_ab.mean())

# save res

In [ ]:
DataFrame(
    {
        "avg_e1": avg_e1,
        "avg_e2": avg_e2,
        "avg_e1_ra": avg_e1_ra,
        "avg_e2_ra": avg_e2_ra,
        "avg_e1_ab": avg_e1_ab,
        "avg_e2_ab": avg_e2_ab,
        "avg_e1_quad": avg_e1_quad,
        "avg_e2_quad": avg_e2_quad,
    },
    schema=[
        "avg_e1",
        "avg_e2",
        "avg_e1_ra",
        "avg_e2_ra",
        "avg_e1_ab",
        "avg_e2_ab",
        "avg_e1_quad",
        "avg_e2_quad",
    ],
).write_csv(VERSION_DIR / "pred_avg_ellips.csv")

del (
    df,
    avg_e1,
    avg_e2,
    avg_e1_ra,
    avg_e2_ra,
    avg_e1_ab,
    avg_e2_ab,
    e1_ra,
    e2_ra,
    e1_ab,
    e2_ab,
    e1_quad,
    e2_quad,
)

# postprocess

In [ ]:
true_galaxies_params = read_csv(
    DATASET_DIR / "true_galaxies_params.csv", separator=",", skip_rows=0
)
true_images_params = read_csv(DATASET_DIR / "true_images_params.csv", separator=",", skip_rows=0)
pred_avg_e = read_csv(VERSION_DIR / "pred_avg_ellips.csv", separator=",", skip_rows=0).to_numpy()

true_avg_e = true_images_params["avg_e1", "avg_e2"].to_numpy()
true_avg_shears = true_images_params["avg_g1", "avg_g2"].to_numpy()
true_e = true_galaxies_params["ell_1", "ell_2"].to_numpy()
print(pred_avg_e.shape, true_avg_e.shape, true_avg_shears.shape)

for i in range(2):
    pred_avg_e = pred_avg_e[numpy.abs(true_avg_shears[:, i]) <= 0.05]
    true_avg_e = true_avg_e[numpy.abs(true_avg_shears[:, i]) <= 0.05]
    true_avg_shears = true_avg_shears[numpy.abs(true_avg_shears[:, i]) <= 0.05]
print("restricted range", pred_avg_e.shape, true_avg_e.shape, true_avg_shears.shape)

# histo

In [ ]:
bins = 10
histos(
    diff_e1=pred_avg_e[:, 0] - pred_avg_e[:, 2],
    diff_e2=pred_avg_e[:, 1] - pred_avg_e[:, 3],
    data_gal=detections,
    directory=PLOTS_DIR / "errors_ra.pdf",
    bins=bins,
)
histos(
    diff_e1=pred_avg_e[:, 6] - pred_avg_e[:, 4],
    diff_e2=pred_avg_e[:, 7] - pred_avg_e[:, 5],
    data_gal=detections,
    directory=PLOTS_DIR / "errors_ab.pdf",
    bins=bins,
)

# best results

In [ ]:
best_results(true_avg_e, true_avg_shears, PLOTS_DIR / "best_results.pdf")

# results e

In [ ]:
case = "comp_ell_normal"
case_results(pred_avg_e[:, 0:2], true_avg_e, PLOTS_DIR / f"results_{case}.pdf", symbol="epsilon")

case = "comp_ell_ab_theta"
case_results(pred_avg_e[:, 4:6], true_avg_e, PLOTS_DIR / f"results_{case}.pdf", symbol="epsilon")

# results e/g

In [ ]:
case = "normal"
case_results(pred_avg_e[:, 0:2], true_avg_shears, PLOTS_DIR / f"results_{case}.pdf")

case = "ratio_angle"
case_results(pred_avg_e[:, 2:4], true_avg_shears, PLOTS_DIR / f"results_{case}.pdf")

In [ ]:
case = "ab_theta"
case_results(pred_avg_e[:, 4:6], true_avg_shears, PLOTS_DIR / f"results_{case}.pdf")

case = "quad"
case_results(pred_avg_e[:, 6:8], true_avg_shears, PLOTS_DIR / f"results_{case}.pdf")

# sersic (numerical stability?)

In [ ]:
# from utils import plot_sersic
# X_min, X_max = -2.5, 1.5
# plot_sersic(X_min, X_max, PLOTS_DIR)
# del X_min, X_max

# psf

In [ ]:
# from utils import check_psf
# check_psf(
#     Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid") / "PSF" / "airy.psf.fits", PLOTS_DIR
# )